In [10]:
# Import data manipulation and visualization libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import filter warnings
import warnings
warnings.filterwarnings("ignore")

# Import OrderedDict for maintaining the order of columns
from collections import OrderedDict

# Import logging
import logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(message)s - %(levelname)s',
                    filemode='w',
                    filename='Classification.log',
                    force=True)

# Import machine learning libraries
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# SMOTE
from imblearn.over_sampling import SMOTE


def data_ingestion(data_source: str) -> pd.DataFrame:
    logging.info("Data Ingestion Started...")
    df = pd.read_csv(data_source)
    logging.info("Data Ingestion Completed Successfully")
    return df


def split_data(data, target_col, test_size=0.3, random_state=42):
    X = data.drop(columns=[target_col])
    y = data[target_col]

    return train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )


def encode_categorical(X_train, X_test):
    X_train = X_train.copy()
    X_test = X_test.copy()

    cat_cols = X_train.select_dtypes(include="object").columns
    encoders = {}

    for col in cat_cols:
        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train[col])
        X_test[col] = X_test[col].map(
            lambda x: le.transform([x])[0] if x in le.classes_ else -1
        )
        encoders[col] = le

    return X_train, X_test, encoders


def train_evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc


def compare_models(X_train, X_test, y_train, y_test):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Decision Tree": DecisionTreeClassifier(),
        "SVC": SVC(),
        "KNN": KNeighborsClassifier(),
        "Random Forest": RandomForestClassifier(),
        "Gradient Boost": GradientBoostingClassifier(),
        "Ada Boost": AdaBoostClassifier(),
        "XG Boost": XGBClassifier(eval_metric='mlogloss')
    }

    results = []

    for name, model in models.items():
        acc = train_evaluate_model(model, X_train, X_test, y_train, y_test)
        results.append([name, acc])

    return pd.DataFrame(
        results, columns=["Model Name", "Accuracy"]
    ).sort_values("Accuracy", ascending=False)


def k_fold_cv(X_train, y_train, folds=10):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Decision Tree": DecisionTreeClassifier(),
        "SVC": SVC(),
        "KNN": KNeighborsClassifier(),
        "Random Forest": RandomForestClassifier(),
        "Gradient Boost": GradientBoostingClassifier(),
        "Ada Boost": AdaBoostClassifier(),
        "XG Boost": XGBClassifier(eval_metric='mlogloss')
    }

    results = []

    for name, model in models.items():
        scores = cross_val_score(
            model, X_train, y_train, cv=folds, scoring="accuracy"
        )
        results.append([name, scores.mean(), scores.std()])

    return pd.DataFrame(
        results, columns=["Model Name", "CV Mean Accuracy", "CV STD"]
    ).sort_values("CV Mean Accuracy", ascending=False)


def hyperparameter_tuning(X_train, y_train, folds=5):
    tuning_config = {
        "XG Boost": {
            "model": XGBClassifier(eval_metric='mlogloss'),
            "params": {
                "max_depth": [3, 5, 7],
                "learning_rate": [0.1, 0.2],
                "gamma": [0, 5]
            }
        },
        "Random Forest": {
            "model": RandomForestClassifier(),
            "params": {
                "max_depth": [5, 10, 15],
                "max_features": ["sqrt", "log2"]
            }
        }
    }

    best_models = {}

    for name, cfg in tuning_config.items():
        grid = GridSearchCV(
            cfg["model"],
            cfg["params"],
            cv=folds,
            scoring="accuracy",
            n_jobs=-1
        )
        grid.fit(X_train, y_train)
        best_models[name] = grid.best_estimator_

    return best_models


def post_tuning_cv(best_models, X_train, y_train, folds=5):
    results = []

    for name, model in best_models.items():
        scores = cross_val_score(
            model, X_train, y_train, cv=folds, scoring="accuracy"
        )
        results.append([name, scores.mean(), scores.std()])

    return pd.DataFrame(
        results, columns=["Model Name", "CV Mean Accuracy", "CV STD"]
    ).sort_values("CV Mean Accuracy", ascending=False)


def final_test_evaluation(best_model, X_train, X_test, y_train, y_test):
    best_model.fit(X_train, y_train)
    y_pred = best_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc


DATA_URL = "https://raw.githubusercontent.com/chandanc5525/CardioVascularRisk_AssessmentModel/refs/heads/main/data/raw/cardiovascular_risk_dataset.csv"
TARGET_COL = "risk_category"


def main():
    logging.info("ML Classification Pipeline Started")

    df = data_ingestion(DATA_URL)

    # Train-Test Split
    X_train, X_test, y_train, y_test = split_data(
        data=df,
        target_col=TARGET_COL,
        test_size=0.3,
        random_state=42
    )

    # Encode Features
    X_train, X_test, encoders = encode_categorical(X_train, X_test)

    # Encode Target (Fix for XGBoost error)
    target_encoder = LabelEncoder()
    y_train = target_encoder.fit_transform(y_train)
    y_test = target_encoder.transform(y_test)

    # Apply SMOTE ONLY on training data
    smote = SMOTE(random_state=42)
    X_train, y_train = smote.fit_resample(X_train, y_train)

    # Baseline Models
    baseline_results = compare_models(
        X_train, X_test, y_train, y_test
    )

    print("\nBaseline Model Comparison:")
    print(baseline_results)

    # Cross Validation
    cv_results = k_fold_cv(
        X_train, y_train, folds=10
    )

    print("\nCross Validation Results (Before Tuning):")
    print(cv_results)

    # Hyperparameter Tuning
    best_models = hyperparameter_tuning(
        X_train, y_train, folds=5
    )

    post_cv_results = post_tuning_cv(
        best_models, X_train, y_train, folds=5
    )

    print("\nCross Validation Results (After Tuning):")
    print(post_cv_results)

    best_model_name = post_cv_results.iloc[0]["Model Name"]
    best_model = best_models[best_model_name]

    final_acc = final_test_evaluation(
        best_model, X_train, X_test, y_train, y_test
    )

    print("\nFinal Test Performance:")
    print(f"Best Model : {best_model_name}")
    print(f"Accuracy   : {final_acc}")

    logging.info("ML Classification Pipeline Completed Successfully")

    return best_model


if __name__ == "__main__":
    main()


Baseline Model Comparison:
            Model Name  Accuracy
1        Decision Tree  1.000000
6            Ada Boost  1.000000
5       Gradient Boost  1.000000
4        Random Forest  1.000000
7             XG Boost  0.998182
0  Logistic Regression  0.976364
3                  KNN  0.473939
2                  SVC  0.461212

Cross Validation Results (Before Tuning):
            Model Name  CV Mean Accuracy    CV STD
1        Decision Tree          0.999575  0.001274
5       Gradient Boost          0.999575  0.001274
6            Ada Boost          0.999575  0.001274
4        Random Forest          0.999151  0.001408
7             XG Boost          0.997879  0.002120
0  Logistic Regression          0.983027  0.006571
3                  KNN          0.552316  0.031964
2                  SVC          0.507956  0.017838

Cross Validation Results (After Tuning):
      Model Name  CV Mean Accuracy    CV STD
1  Random Forest          0.999363  0.000849
0       XG Boost          0.997667  0.001